# 0. Imports

In [1]:
#!pip install -qq ipython numpy pandas scikit-learn statsmodels xgboost torch

In [2]:
import joblib, json, sys, warnings
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import sklearn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import torch
from torch import nn
from torch.utils.data import Dataset, TensorDataset, DataLoader

In [3]:
!python --version
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)

Python 3.10.18
numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2
torch 2.9.1


# 1. Preprocessing

In [4]:
# AUTOREGRESSIVE (LAG) FEATURES

# lags = sorted(set(
#     list(range(1, 4)) + [6, 12, 18, 24, 36, 48, 72, 168]
#     + list(range(24, 27)) + [36, 48]
#     + [24*i for i in range(3,7)]
#     + [24*7*i for i in range(1,5)]
# ))
lags = [1, 2, 3, 6, 12, 18, 24, 36, 48, 72, 168]
lagFeatures = [f"Adjusted demand -{h} hr" for h in lags]

In [5]:
# CALENDAR FEATURES

# Raw integer calendar features
intDateTimeFeatures = ["Hour", "Month", "DayOfWeek", "DayOfYear"]

# Low order hour of day and day of year Fourier term features
hourFourierFeatures, dayFourierFeatures = [], []
for i in (1, 2, 3):
    argStr = (f"{i}*" if i>1 else "") + "Hour"
    hourFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])
    argStr = (f"{i}*" if i>1 else "") + "DayOfYear"
    dayFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])

# Hour of day and day of week one-hot encodings.
# Weekend and holiday flags.
hourDummyFeatures = [f"Hour_Flag_{h}" for h in range(24)]
dayDummyFeatures = ["Day_Flag_Weekend", "Day_Flag_Holiday"]
dayDummyFeatures += [f"DayOfWeek_Flag_{d}" for d in range(7)]
monthDummyFeatures = [f"Month_Flag_{m}" for m in range(1, 13)]

# Collate all calendar features
calendarFeatures = (
    # intDateTimeFeatures
    hourFourierFeatures
    + dayFourierFeatures
    + dayDummyFeatures[:2]
)


In [6]:
# ENERGY FEATURES

energyFeatures = [
    # "Adjusted net generation",
    # "Adjusted total interchange",
    "FPC", "FMPP", "SOCO", "TEC",
    "JEA", "SEC", "HST", "GVL",
]


In [7]:
# WEATHER FEATURES

skyCodes = ['BKN', 'CLR', 'FEW', 'SCT', 'OVC', 'NA']
directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW", "VRB"]

weatherFeatures = ([
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlySeaLevelPressure",
    "HourlyVisibility",
    "HourlyWindSpeed"]
    + [f"HourlySkyConditions_Flag_{code}" for code in skyCodes]
    + ["HourlyWindDirection_Flag_VRB",]
    + ["sin(HourlyWindDirection)", "cos(HourlyWindDirection)"]
)


In [8]:
# Define other convenient feature variables.

allFeatures = (
    lagFeatures 
    + calendarFeatures 
    + energyFeatures 
    + weatherFeatures
)
selectedFeatures = ['Day_Flag_Weekend', 'Day_Flag_Holiday', 'sin(DayOfYear)', 'HourlySeaLevelPressure', 'HourlyWindSpeed', 'sin(HourlyWindDirection)', 'cos(HourlyWindDirection)', 'Adjusted demand -6 hr', 'HourlyRelativeHumidity', 'Adjusted demand -12 hr', 'Adjusted demand -1 hr', 'SEC', 'GVL', 'cos(DayOfYear)', 'FMPP', 'TEC', 'JEA', 'cos(2*DayOfYear)', 'HourlyPrecipitation', 'HourlySkyConditions_Flag_OVC', 'HourlyVisibility', 'HourlySkyConditions_Flag_SCT', 'HourlySkyConditions_Flag_BKN', 'HourlySkyConditions_Flag_NA', 'HourlySkyConditions_Flag_CLR', 'HourlyWindDirection_Flag_VRB', 'sin(2*DayOfYear)', 'sin(3*DayOfYear)', 'cos(3*DayOfYear)']

### 1.3 Data Splits

In [9]:
# reproducibility
np.random.seed(42)
torch.manual_seed(42)

# --- paths ---
data_root   = Path("data/clean")
train_dir   = data_root / "train"
val_dir     = data_root / "val"

DFtrain = pd.read_pickle(train_dir / "DFtrain.pkl")
DFval   = pd.read_pickle(val_dir   / "DFval.pkl")

# --- define target and features (ADJUST target name to yours) ---
TARGET_COL   = "Adjusted demand"   # <--- change to your target column
TIME_COL     = "t"        # if you have a time column


# 2. LSTM

In [10]:
# --- peak weighting setup for LSTM model_F (reuse these if defined earlier) ---

# 75th percentile threshold, computed on TRAIN ONLY
q75 = DFtrain[TARGET_COL].quantile(0.75)
print(f"75th percentile training threshold for {TARGET_COL}: {q75:.3f}")

# choose how much more to weight peak hours
peak_weight = 3.0  # e.g. 2.0, 3.0, 5.0


75th percentile training threshold for Adjusted demand: 17395.000


In [11]:
MODEL_SPECS = {
    "model_A": {
        "feature_cols": lagFeatures,
    },
    "model_B": {
        "feature_cols": lagFeatures + calendarFeatures,
    },
    "model_C": {
        "feature_cols": lagFeatures + calendarFeatures + energyFeatures,
    },
    "model_D": {
        "feature_cols": lagFeatures + calendarFeatures + energyFeatures + weatherFeatures,
    },
    "model_E": {
        "feature_cols": selectedFeatures,
    },
    "model_F": {
        "feature_cols": allFeatures,
    },
}

data_root   = Path("data/clean")
train_dir   = data_root / "train"
val_dir     = data_root / "val"

train_block_paths = sorted(train_dir.glob("DFtrain_block*.pkl"))
val_block_paths   = sorted(val_dir.glob("DFval_block*.pkl"))

print("Train blocks:", len(train_block_paths), " Val blocks:", len(val_block_paths))
print("n_train DF:", len(DFtrain), "n_val DF:", len(DFval))

# where to save LSTM stuff
lstm_root = Path("models") / "LSTM"
lstm_root.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==================================================
# 2. Sequence helpers (unchanged logic, more general)
# ==================================================

def make_seq(X, y, seq_len):
    """
    Build overlapping (sequence, target) pairs from one continuous block.
    X: (N, num_features), y: (N,)
    returns:
      X_seq: (N - seq_len, seq_len, num_features)
      y_seq: (N - seq_len,)
    """
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32).reshape(-1)

    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])

    if not Xs:
        return None, None

    return torch.tensor(np.stack(Xs)), torch.tensor(np.array(ys))


def make_seq_from_blocks(block_paths, seq_len, df_to_scaled_arrays_fn):
    """
    For a list of block .pkl files, build sequences per block and concatenate.
    Skips blocks that are too short for at least one full sequence.
    df_to_scaled_arrays_fn: function mapping df -> (X_s, y_s)
    """
    X_seq_list = []
    y_seq_list = []

    for path in block_paths:
        df_block = pd.read_pickle(path)

        if len(df_block) <= seq_len:
            print(f"Skipping {path.name}: length {len(df_block)} <= seq_len={seq_len}")
            continue

        X_s, y_s = df_to_scaled_arrays_fn(df_block)
        X_block, y_block = make_seq(X_s, y_s, seq_len)

        if X_block is None:
            print(f"Skipping {path.name}: no sequences produced")
            continue

        X_seq_list.append(X_block)
        y_seq_list.append(y_block)

    if not X_seq_list:
        raise ValueError(
            f"No blocks produced sequences; ensure some blocks have length > seq_len={seq_len}."
        )

    X_seq_all = torch.cat(X_seq_list, dim=0)
    y_seq_all = torch.cat(y_seq_list, dim=0)
    return X_seq_all, y_seq_all

# ==================================================
# 3. LSTM model definition
# ==================================================

class LSTMRegressor(nn.Module):
    """
    LSTM-based regressor that maps a sequence of feature vectors
    to a single scalar prediction (next-hour demand).
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch_size, seq_len, num_features)
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]   # last time step
        output = self.fc(last_hidden)      # (batch_size, 1)
        return output.squeeze(-1)          # (batch_size,)

# ==================================================
# 4. Train + validate ONE LSTM config (generic)
# ==================================================

def train_lstm_and_eval(
    hidden_size,
    num_layers,
    lr,
    n_epochs,
    input_size,
    train_loader,
    val_loader,
    device,
    y_scaler,
    dropout=0.2,
    use_peak_weights=False,
):
    """
    Train one LSTM configuration and return final val RMSE (original units) + trained model.

    If use_peak_weights=True, train_loader and val_loader must yield (X, y, w)
    where w is a 1D tensor of sample weights. Training uses weighted MSE in scaled
    space; validation uses peak-weighted RMSE in original units.
    """
    model = LSTMRegressor(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss(reduction="none")  # we'll handle reduction ourselves if weighted

    for epoch in range(1, n_epochs + 1):
        # --- train epoch ---
        model.train()
        batch_losses = []

        for batch in train_loader:
            if use_peak_weights:
                X_batch, y_batch, w_batch = batch
                w_batch = w_batch.to(device)
            else:
                X_batch, y_batch = batch
                w_batch = None

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            y_pred_batch = model(X_batch)

            # MSE in scaled y-space
            se = (y_pred_batch - y_batch) ** 2  # (batch,)

            if use_peak_weights and w_batch is not None:
                # weighted MSE: sum(w * se) / sum(w)
                loss = (w_batch * se).sum() / w_batch.sum()
            else:
                loss = se.mean()

            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

        # --- validation at end of epoch ---
        model.eval()
        val_preds_scaled = []
        val_true_scaled  = []
        val_weights_all  = [] if use_peak_weights else None

        with torch.no_grad():
            for batch in val_loader:
                if use_peak_weights:
                    X_batch, y_batch, w_batch = batch
                    w_batch = w_batch.to(device)
                else:
                    X_batch, y_batch = batch
                    w_batch = None

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                y_pred = model(X_batch)

                val_preds_scaled.append(y_pred.cpu().numpy())
                val_true_scaled.append(y_batch.cpu().numpy())
                if use_peak_weights and val_weights_all is not None:
                    val_weights_all.append(w_batch.cpu().numpy())

        y_pred_s = np.concatenate(val_preds_scaled)   # scaled y_hat
        y_true_s = np.concatenate(val_true_scaled)    # scaled y

        # back to original units
        y_pred = y_scaler.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
        y_true = y_scaler.inverse_transform(y_true_s.reshape(-1, 1)).ravel()

        if use_peak_weights and val_weights_all is not None:
            w_val = np.concatenate(val_weights_all).astype(float)
            val_rmse = np.sqrt(np.sum(w_val * (y_true - y_pred) ** 2) / np.sum(w_val))
        else:
            val_rmse = root_mean_squared_error(y_true, y_pred)

        print(
            f"[LSTM hs={hidden_size}, layers={num_layers}, lr={lr}] "
            f"Epoch {epoch:02d}  Train MSE (scaled)={np.mean(batch_losses):.3f}  "
            f"Val RMSE={val_rmse:.2f}"
        )

    return val_rmse, model


# ==================================================
# 5. Hyperparameter search PER CATEGORY + saving
# ==================================================

seq_len = 24  # past 24 hours -> next hour

hidden_sizes    = [32, 64]
num_layers_list = [1, 2]
learning_rates  = [0.01, 0.001]
n_epochs        = 10

lstm_results = {}

for model_name, spec in MODEL_SPECS.items():
    feature_cols = spec["feature_cols"]
    print(f"\n==============================")
    print(f"=== LSTM for {model_name} ===")
    print("n_features:", len(feature_cols))

    # --- split into numeric vs flag/dummy cols for THIS category ---
    flag_cols = [
        c for c in feature_cols
        if ("Flag" in c) or (DFtrain[c].dtype in ("bool", "boolean"))
    ]
    num_cols = [c for c in feature_cols if c not in flag_cols]

    print("  numeric cols:", len(num_cols), " flag cols:", len(flag_cols))

    # --- scalers for THIS category (fit on numeric cols only) ---
    x_scaler_lstm = StandardScaler()
    y_scaler_lstm = StandardScaler()

    Xnum_train_all = DFtrain[num_cols].to_numpy()
    y_train_all    = DFtrain[TARGET_COL].to_numpy()

    x_scaler_lstm.fit(Xnum_train_all)
    y_scaler_lstm.fit(y_train_all.reshape(-1, 1))

    # local scaling helper uses this category's numeric + flag sets + scalers
    def df_to_scaled_arrays_local(df):
        Xnum = df[num_cols].to_numpy()
        Xflg = df[flag_cols].to_numpy().astype(np.float32)  # keep 0/1 as-is
        y    = df[TARGET_COL].to_numpy()

        Xnum_s = x_scaler_lstm.transform(Xnum).astype(np.float32)
        X_s = np.concatenate([Xnum_s, Xflg], axis=1)

        y_s = y_scaler_lstm.transform(y.reshape(-1, 1)).ravel().astype(np.float32)
        return X_s, y_s

    # --- build sequences from blocks ---
    Xtr_seq, ytr_seq   = make_seq_from_blocks(train_block_paths, seq_len, df_to_scaled_arrays_local)
    Xval_seq, yval_seq = make_seq_from_blocks(val_block_paths,   seq_len, df_to_scaled_arrays_local)

    print("Train sequences:", Xtr_seq.shape)
    print("Val sequences  :", Xval_seq.shape)

    # --- build weights for sequences if this is model_F ---
    if model_name == "model_F":
        # Convert sequence targets back to original units
        ytr_orig = y_scaler_lstm.inverse_transform(
            ytr_seq.numpy().reshape(-1, 1)
        ).ravel()
        yval_orig = y_scaler_lstm.inverse_transform(
            yval_seq.numpy().reshape(-1, 1)
        ).ravel()

        peak_mask_tr_seq  = ytr_orig >= q75
        peak_mask_val_seq = yval_orig >= q75

        w_tr  = np.where(peak_mask_tr_seq,  peak_weight, 1.0).astype(np.float32)
        w_val = np.where(peak_mask_val_seq, peak_weight, 1.0).astype(np.float32)

        train_dataset_lstm = TensorDataset(
            Xtr_seq,
            ytr_seq,
            torch.tensor(w_tr, dtype=torch.float32),
        )
        val_dataset_lstm = TensorDataset(
            Xval_seq,
            yval_seq,
            torch.tensor(w_val, dtype=torch.float32),
        )

        use_peak_weights = True
        print("  Using peak-weighted training and validation loss for LSTM model_F.")
    else:
        # no weights for A–E
        train_dataset_lstm = TensorDataset(Xtr_seq, ytr_seq)
        val_dataset_lstm   = TensorDataset(Xval_seq, yval_seq)
        use_peak_weights   = False

    train_loader_lstm = DataLoader(
        train_dataset_lstm,
        batch_size=128,
        shuffle=True,
    )
    val_loader_lstm = DataLoader(
        val_dataset_lstm,
        batch_size=128,
        shuffle=False,
    )

    input_size = Xtr_seq.shape[2]

    # --- tiny hyperparameter search for THIS category ---
    best_cfg   = None
    best_rmse  = np.inf
    best_model = None

    for hs in hidden_sizes:
        for nl in num_layers_list:
            for lr in learning_rates:
                print(f"\n=== {model_name} config: hs={hs}, layers={nl}, lr={lr} ===")
                val_rmse, model = train_lstm_and_eval(
                    hidden_size=hs,
                    num_layers=nl,
                    lr=lr,
                    n_epochs=n_epochs,
                    input_size=input_size,
                    train_loader=train_loader_lstm,
                    val_loader=val_loader_lstm,
                    device=device,
                    y_scaler=y_scaler_lstm,
                    dropout=0.2,
                    use_peak_weights=use_peak_weights,
                )
                if val_rmse < best_rmse:
                    best_rmse  = val_rmse
                    best_cfg   = dict(hidden_size=hs, num_layers=nl, lr=lr)
                    best_model = model

    print(f"\nBest LSTM for {model_name}: cfg={best_cfg}  (val RMSE={best_rmse:.2f})")

    # store summary
    lstm_results[model_name] = {
        "best_cfg": best_cfg,
        "best_rmse": best_rmse,
        "feature_cols": list(feature_cols),
    }

    # --- save best model + scalers + meta for THIS category ---
    best_model_cpu = best_model.to("cpu")  # store on CPU for portability
    
    ckpt = {
        "model": best_model_cpu,                 # full nn.Module
        "feature_cols": list(feature_cols),
        "target_col": TARGET_COL,
        "val_rmse": float(best_rmse),
        "x_scaler": x_scaler_lstm,
        "y_scaler": y_scaler_lstm,
        "seq_len": seq_len,
    }
    if model_name == "model_F":
        ckpt["peak_weighting"] = {
            "quantile": 0.75,
            "threshold": float(q75),
            "peak_weight": float(peak_weight),
        }
    
    ckpt_path = lstm_root / f"{model_name}_LSTM_best.pt"
    torch.save(ckpt, ckpt_path)
    print(f"Saved best {model_name} LSTM checkpoint to: {ckpt_path}")
    
    # if you still want to keep it on GPU after saving:
    best_model = best_model.to(device)

print("\nDone: best validated LSTM for EACH category (A–F) has been trained and saved.")


Train blocks: 142  Val blocks: 31
n_train DF: 59647 n_val DF: 12782
Using device: cpu

=== LSTM for model_A ===
n_features: 11
  numeric cols: 11  flag cols: 0
Skipping DFtrain_block010.pkl: length 23 <= seq_len=24
Skipping DFtrain_block019.pkl: length 22 <= seq_len=24
Skipping DFtrain_block025.pkl: length 24 <= seq_len=24
Skipping DFtrain_block027.pkl: length 23 <= seq_len=24
Skipping DFtrain_block031.pkl: length 23 <= seq_len=24
Skipping DFtrain_block034.pkl: length 23 <= seq_len=24
Skipping DFtrain_block042.pkl: length 5 <= seq_len=24
Skipping DFtrain_block045.pkl: length 18 <= seq_len=24
Skipping DFtrain_block050.pkl: length 23 <= seq_len=24
Skipping DFtrain_block054.pkl: length 23 <= seq_len=24
Skipping DFtrain_block057.pkl: length 2 <= seq_len=24
Skipping DFtrain_block069.pkl: length 23 <= seq_len=24
Skipping DFtrain_block072.pkl: length 3 <= seq_len=24
Skipping DFtrain_block074.pkl: length 17 <= seq_len=24
Skipping DFtrain_block081.pkl: length 20 <= seq_len=24
Skipping DFtrain_b